<a href="https://colab.research.google.com/github/Foysal-Munsy/messy-trajectory-evaluation/blob/main/01_dataset_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Compatibility Check

Goal: Verify that the coding-agent trajectory dataset is compatible
with SWE-ABS strengthened evaluation.

Trajectory dataset:
tarsur385/swe-verified-gemini3-flash-trajectories

Strengthened-test dataset:
OpenAgentLab/SWE-Bench_Verified_ABS

In [2]:
!pip install -q datasets pandas

In [3]:
from datasets import load_dataset

traj = load_dataset(
    "tarsur385/swe-verified-gemini3-flash-trajectories",
    split="train"
)
from datasets import load_dataset



swe_abs = load_dataset(
    "OpenAgentLab/SWE-Bench_Verified_ABS",
    split="test"
)

print("Trajectory rows:", len(traj))
print("SWE-ABS rows:", len(swe_abs))

README.md:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

data.jsonl: reconstructing file:   0%|          |  0.00B /  237MB            

data.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/296 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/3.58k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.17MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Trajectory rows: 296
SWE-ABS rows: 500


In [4]:
print("Trajectory columns:")
print(traj.column_names)

print("\nSWE-ABS columns:")
print(swe_abs.column_names)

Trajectory columns:
['instance_id', 'sample', 'model', 'reasoning_effort', 'temperature', 'resolved', 'n_steps', 'num_messages', 'patch', 'messages']

SWE-ABS columns:
['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'difficulty', 'original_test_patch']


In [5]:
traj_ids = set(traj["instance_id"])
abs_ids = set(swe_abs["instance_id"])

overlap = traj_ids & abs_ids
missing = traj_ids - abs_ids

print("Unique trajectory tasks:", len(traj_ids))
print("Tasks found in SWE-ABS:", len(overlap))
print("Tasks missing from SWE-ABS:", len(missing))

if missing:
    print("\nMissing instance IDs:")
    print(sorted(missing))

Unique trajectory tasks: 100
Tasks found in SWE-ABS: 100
Tasks missing from SWE-ABS: 0


# PASS trajectories filter

In [6]:
import pandas as pd

traj_df = traj.to_pandas()

print(traj_df["resolved"].value_counts(dropna=False))

resolved
True     198
False     98
Name: count, dtype: int64


In [7]:
pass_df = traj_df[traj_df["resolved"] == True].copy()

print("Total trajectories:", len(traj_df))
print("Standard PASS trajectories:", len(pass_df))
print("Unique tasks among PASS trajectories:", pass_df["instance_id"].nunique())

Total trajectories: 296
Standard PASS trajectories: 198
Unique tasks among PASS trajectories: 73


In [8]:
print(
    pass_df.groupby("instance_id").size().value_counts().sort_index()
)

1     6
2     9
3    58
Name: count, dtype: int64


In [9]:
example = pass_df.iloc[0]

print("Instance ID:", example["instance_id"])
print("Sample:", example["sample"])
print("n_steps:", example["n_steps"])
print("Number of messages:", len(example["messages"]))
print("Messages type:", type(example["messages"]))

print("\nFirst 3 messages:\n")

for i, msg in enumerate(example["messages"][:3]):
    print(f"\n--- Message {i} ---")
    print(msg)

Instance ID: astropy__astropy-12907
Sample: 1
n_steps: 50
Number of messages: 102
Messages type: <class 'numpy.ndarray'>

First 3 messages:


--- Message 0 ---
{"role":"system","content":"You are OpenHands agent, a helpful AI assistant that can interact with a computer to solve tasks.\n\n<ROLE>\n* Your primary role is to assist users by executing commands, modifying code, and solving technical problems effectively. You should be thorough, methodical, and prioritize quality over speed.\n* If the user asks a question, like \"why is X happening\", don't try to fix the problem. Just give an answer to the question.\n<\/ROLE>\n\n<MEMORY>\n* Use `AGENTS.md` under the repository root as your persistent memory for repository-specific knowledge and context.\n* Add important insights, patterns, and learnings to this file to improve future task performance.\n* This repository skill is automatically loaded for every conversation and helps maintain context across sessions.\n* For more information ab